In [3]:
from google_play_scraper import app,reviews,Sort, reviews_all
import pandas as pd
import pyspark
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
scraperReview = reviews_all(
    "com.mobile.legends", # id aplikasi di play store
    lang = 'id', # bahasa ulasan
    country = "id", # negara
    sort = Sort.MOST_RELEVANT, # urutan ulasan
    count = 20000, # jumlah maks data yangg akan diambil
)

In [5]:
scraperReview

[{'reviewId': 'a88d9682-8a77-433a-825c-493070197e3d',
  'userName': 'Pengguna Google',
  'userImage': 'https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g',
  'content': 'Aku turunkan rating bintang nya. Tolong pelaku cheat dan RW di blacklist. Sering mengalami pas masuk in game tiba² ping naik dan tidak bisa lanjut main, bahakan tidak bisa masuk in game nya. Padahal jaringan Aman² saja. Tolong diperketat lagi untuk keamanannya demi kenyamanan bermain.',
  'score': 1,
  'thumbsUpCount': 5153,
  'reviewCreatedVersion': '1.9.48.10373',
  'at': datetime.datetime(2025, 3, 8, 9, 16, 20),
  'replyContent': None,
  'repliedAt': None,
  'appVersion': '1.9.48.10373'},
 {'reviewId': '97add647-794f-42eb-a02e-fd58f5a424a5',
  'userName': 'Pengguna Google',
  'userImage': 'https://play-lh.googleusercontent.com/EGemoI2NTXmTsBVtJqk8jxF9rh8ApRWfsIMQSt2uE4OcpQqbFu7f7NbTK05lx80nuSijCz7sc3a277R67g',
  'content': 'Game nya diperbarui lag

In [6]:
import csv

with open("Review ML.csv", mode = 'w', newline='', encoding = 'utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Review", "Rating"])
    for review in scraperReview:
        writer.writerow([review['content'],review['score']])

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ReviewML").getOrCreate()

In [11]:
spark

In [65]:
df= spark.read.csv("Review ML.csv", header= True, inferSchema= True)
df.show()

+--------------------+------+
|              Review|Rating|
+--------------------+------+
|Aku turunkan rati...|     1|
|Game nya diperbar...|     1|
|Kondisi lag saat ...|     1|
|Montonnnnn, giman...|     3|
|Sama saja kasih s...|     1|
|jujur ya jujur sa...|     1|
|Untuk mode rank s...|     1|
|Kesel, udah serin...|     1|
|Game gajelas, kay...|     1|
|Semakin kesini ml...|     4|
|Event nya banyak+...|     2|
|Anehhh, tolong pe...|     1|
|Pr tambahan moont...|     1|
|woi min benerin l...|     1|
|Banyak yang make ...|     5|
|Jgn senang dlu di...|     5|
|Terlalu banyak bu...|     1|
|"sebelumnya asik ...|     4|
|Dasar game sampah...|     1|
|Game jelek plicc,...|     1|
+--------------------+------+
only showing top 20 rows



In [66]:
df.printSchema()

root
 |-- Review: string (nullable = true)
 |-- Rating: string (nullable = true)



In [67]:
df.columns

['Review', 'Rating']

In [68]:
df.dtypes

[('Review', 'string'), ('Rating', 'string')]

In [69]:
from pyspark.sql.types import IntegerType

df = df.withColumn("Rating", df['Rating'].cast(IntegerType()))

In [70]:
df.printSchema()

root
 |-- Review: string (nullable = true)
 |-- Rating: integer (nullable = true)



In [71]:
df.show()

+--------------------+------+
|              Review|Rating|
+--------------------+------+
|Aku turunkan rati...|     1|
|Game nya diperbar...|     1|
|Kondisi lag saat ...|     1|
|Montonnnnn, giman...|     3|
|Sama saja kasih s...|     1|
|jujur ya jujur sa...|     1|
|Untuk mode rank s...|     1|
|Kesel, udah serin...|     1|
|Game gajelas, kay...|     1|
|Semakin kesini ml...|     4|
|Event nya banyak+...|     2|
|Anehhh, tolong pe...|     1|
|Pr tambahan moont...|     1|
|woi min benerin l...|     1|
|Banyak yang make ...|     5|
|Jgn senang dlu di...|     5|
|Terlalu banyak bu...|     1|
|"sebelumnya asik ...|     4|
|Dasar game sampah...|     1|
|Game jelek plicc,...|     1|
+--------------------+------+
only showing top 20 rows



In [72]:
df.describe().show()

+-------+--------------------+------------------+
|summary|              Review|            Rating|
+-------+--------------------+------------------+
|  count|               90172|             86608|
|   mean|                NULL|2.2840268797339736|
| stddev|                NULL|1.6615780949172498|
|    min|!!!!!Dear Para De...|                 1|
|    max|🤣🤣🤣 mantap moo...|               158|
+-------+--------------------+------------------+



In [73]:
from pyspark.sql.functions import col

df.filter(
    (col("Review").isNull()) |
    (col("Review").isNull())
).show()

+------+------+
|Review|Rating|
+------+------+
+------+------+



In [74]:
df.groupBy("Rating").count().show()

+------+-----+
|Rating|count|
+------+-----+
|  NULL| 3564|
|     1|45148|
|     3| 9080|
|     5|15503|
|     4| 7001|
|     2| 9875|
|   158|    1|
+------+-----+



In [75]:
df = df.dropna()
df.show()

+--------------------+------+
|              Review|Rating|
+--------------------+------+
|Aku turunkan rati...|     1|
|Game nya diperbar...|     1|
|Kondisi lag saat ...|     1|
|Montonnnnn, giman...|     3|
|Sama saja kasih s...|     1|
|jujur ya jujur sa...|     1|
|Untuk mode rank s...|     1|
|Kesel, udah serin...|     1|
|Game gajelas, kay...|     1|
|Semakin kesini ml...|     4|
|Event nya banyak+...|     2|
|Anehhh, tolong pe...|     1|
|Pr tambahan moont...|     1|
|woi min benerin l...|     1|
|Banyak yang make ...|     5|
|Jgn senang dlu di...|     5|
|Terlalu banyak bu...|     1|
|"sebelumnya asik ...|     4|
|Dasar game sampah...|     1|
|Game jelek plicc,...|     1|
+--------------------+------+
only showing top 20 rows



In [76]:
df.describe().show()

+-------+--------------------+------------------+
|summary|              Review|            Rating|
+-------+--------------------+------------------+
|  count|               86608|             86608|
|   mean|                NULL|2.2840268797339736|
| stddev|                NULL|1.6615780949172498|
|    min|!!!!!Dear Para De...|                 1|
|    max|🤣🤣🤣 mantap moo...|               158|
+-------+--------------------+------------------+



In [77]:
df.groupBy("Rating").count().show()

+------+-----+
|Rating|count|
+------+-----+
|     1|45148|
|     3| 9080|
|     5|15503|
|     4| 7001|
|     2| 9875|
|   158|    1|
+------+-----+



In [78]:
dfClean = df.filter(df["Rating"] != 158)
dfClean.show()

+--------------------+------+
|              Review|Rating|
+--------------------+------+
|Aku turunkan rati...|     1|
|Game nya diperbar...|     1|
|Kondisi lag saat ...|     1|
|Montonnnnn, giman...|     3|
|Sama saja kasih s...|     1|
|jujur ya jujur sa...|     1|
|Untuk mode rank s...|     1|
|Kesel, udah serin...|     1|
|Game gajelas, kay...|     1|
|Semakin kesini ml...|     4|
|Event nya banyak+...|     2|
|Anehhh, tolong pe...|     1|
|Pr tambahan moont...|     1|
|woi min benerin l...|     1|
|Banyak yang make ...|     5|
|Jgn senang dlu di...|     5|
|Terlalu banyak bu...|     1|
|"sebelumnya asik ...|     4|
|Dasar game sampah...|     1|
|Game jelek plicc,...|     1|
+--------------------+------+
only showing top 20 rows



In [81]:
dfClean.describe().show()

+-------+--------------------+-----------------+
|summary|              Review|           Rating|
+-------+--------------------+-----------------+
|  count|               86607|            86607|
|   mean|                NULL|2.282228919140485|
| stddev|                NULL|1.575086027759715|
|    min|!!!!!Dear Para De...|                1|
|    max|🤣🤣🤣 mantap moo...|                5|
+-------+--------------------+-----------------+



In [82]:
dfClean.groupBy("Rating").count().show()

+------+-----+
|Rating|count|
+------+-----+
|     1|45148|
|     3| 9080|
|     5|15503|
|     4| 7001|
|     2| 9875|
+------+-----+

